## Guardrails with LangChain

In [22]:
from typing import Any
from langchain_openai import ChatOpenAI
from langchain.agents import create_agent
from langchain.tools import tool
from langchain.messages import HumanMessage, AIMessage
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from langchain.agents.middleware import (
    PIIMiddleware,
    HumanInTheLoopMiddleware,
    AgentMiddleware, 
    AgentState, 
    hook_config
)

from rich import print
import sys
sys.path.append('..')
from utils.helper import print_messages


In [21]:
llm = ChatOpenAI(model="gpt-5-nano")

### Example 1. Guardail with Basic PII Detection to REDACT & Block the Agentic Flow

In [11]:
agent = create_agent(
    model="openai:gpt-5-mini",
    middleware=[
        # Redact emails in user input before sending to model
        PIIMiddleware(
            "email",
            strategy="redact",
            apply_to_input=True,
        ),
        # Mask credit cards in user input
        PIIMiddleware(
            "credit_card",
            strategy="mask",
            apply_to_input=True,
        ),
        # Block API keys - raise error if detected
        PIIMiddleware(
            "api_key",
            detector=r"sk-[a-zA-Z0-9]{32}",
            strategy="block",
            apply_to_input=True,
        ),
    ],
)

**Test PII Redaction**

In [4]:
result = agent.invoke({
    "messages": [HumanMessage(content="My email is shivam@example.com and my card is 5105-1051-0510-5100. Can you help me?")]
})


In [6]:
print_messages(result)

HumanMessage: My email is [REDACTED_EMAIL] and my card is ****-****-****-5100. Can you help me?

================================================================================

Agent: I can help. For safety, please don’t share your full card number. I see your card ends with 5100. What would
you like to do? Here are some common options:

- Reset password or recover account access
- View recent transactions
- Update or verify your payment method
- Update your contact email
- Dispute a charge or review a specific transaction

If you want me to access your account to perform a task, I can look it up using the email on file. Please provide 
the exact email address on your account (you can paste it here), and confirm the action you’d like me to take. I’ll
proceed with a secure lookup and then carry out the requested task.

If you’d rather not share credentials here, I can guide you through the steps to do these tasks directly in your 
account portal or app.

================================================================================

**Test API Key Blocking**

In [15]:
try:
    result = agent.invoke({
        "messages": [HumanMessage(content="Here is my key: sk-abcdefghijklmnopqrstuvwxyz123456")]
    })
except Exception as e:
    print(f"Blocked: {e}")        

Blocked: Detected 1 instance(s) of api_key in text content

### Example 2. Guardrails stack in Agent Execution 
```text
            User Input
                ↓
            [Layer 1] ContentFilterMiddleware    ← Deterministic input filter
                ↓
            [Layer 2] PIIMiddleware (input)      ← PII redaction on input
                ↓
            [Layer 3] HumanInTheLoopMiddleware   ← Approval for sensitive tools
                ↓
            [Layer 4] PIIMiddleware (output)     ← PII redaction on output
                ↓
            [Layer 5] SafetyGuardrailMiddleware  ← Model-based output safety
                ↓
            User Response
```

In [17]:
@tool
def search_tool(query: str) -> str:
    """Search for information."""
    return f"Search results: {query}"

@tool
def send_email_tool(to: str, body: str) -> str:
    """Send an email."""
    return f"Email sent to {to}"

**ContentFilterMiddleware** 
- A deterministic, keyword-based guardrail that runs in `before_agent`, i.e. before the request reaches the LLM.
- Checks only the first human message against a list of `banned_keywords`.
- If a match is found, it short-circuits the run by returning `jump_to: "end"` with a refusal message.
- Because it's a plain string check, it costs zero LLM calls for blocked requests.

In [23]:
class ContentFilterMiddleware(AgentMiddleware):
    """
    Deterministic guardrail: Block requests containing banned keywords.
    This runs BEFORE the agent processes anything — zero LLM cost for blocked requests.
    """

    def __init__(self, banned_keywords: list[str]):
        super().__init__()
        self.banned_keywords = [kw.lower() for kw in banned_keywords]

    @hook_config(can_jump_to=["end"])
    def before_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        first_message = state["messages"][0]
        if first_message.type != "human":
            return None

        content = first_message.content.lower()

        for keyword in self.banned_keywords:
            if keyword in content:
                print(f"🚫 Blocked — keyword detected: '{keyword}'")
                return {"messages": 
                        [AIMessage(content=("I cannot process requests containing inappropriate content. ""Please rephrase your request."))],
                        "jump_to": "end"
                        }
        return None

**SafetyGuardrailMiddleware**
- A model-based guardrail that runs in `after_agent`, i.e. after the LLM produces a response, before it reaches the user.
- Sends the last `AIMessage` content to a smaller, cheaper model (`gpt-5-nano`) with a SAFE/UNSAFE classification prompt.
- If flagged `UNSAFE`, it overwrites the message content with a safe fallback response.
- Costs one extra (cheap) LLM call per agent response.

In [24]:

class SafetyGuardrailMiddleware(AgentMiddleware):
    """
    Model-based guardrail: Use an LLM to evaluate response safety.
    Runs AFTER the agent produces a response, before it reaches the user.
    """

    def __init__(self):
        super().__init__()
        # Use a smaller, cheaper model for the safety check
        self.safety_model = ChatOpenAI(model="gpt-5-nano")

    @hook_config(can_jump_to=["end"])
    def after_agent(self, state: AgentState, runtime: Runtime) -> dict[str, Any] | None:
        if not state["messages"]:
            return None

        last_message = state["messages"][-1]
        if not isinstance(last_message, AIMessage):
            return None

        safety_prompt = f"""Evaluate if this AI response is safe and appropriate for users.Respond with only 'SAFE' or 'UNSAFE'.
        Response to evaluate: 
        {last_message.content}"""

        result = self.safety_model.invoke([{"role": "user", "content": safety_prompt}])

        if "UNSAFE" in result.content.upper():
            print("⚠️  Output flagged as UNSAFE — replacing with safe fallback")
            last_message.content = (
                "I'm unable to provide that response. "
                "Please rephrase your request or contact support."
            )

        return None


**Defining layered Guarded agent**

In [28]:

agent = create_agent(
    model="gpt-5-mini",
    tools=[search_tool, send_email_tool],
    middleware=[
        # Layer 1: Deterministic input filter (before agent)
        ContentFilterMiddleware(banned_keywords=["hack", "exploit", "malware", 'robbery']),

        # Layer 2: PII redaction on input
        PIIMiddleware("credit_card", strategy="mask", apply_to_input=True),

        # Layer 3: Human approval for sensitive tools
        HumanInTheLoopMiddleware(
            interrupt_on={"send_email_tool": True, "search_tool": False}
        ),

        # Layer 4: PII redaction on output
        PIIMiddleware("email", strategy="redact", apply_to_output=True),

        # Layer 5: Model-based output safety
        SafetyGuardrailMiddleware(),
    ],
    checkpointer=InMemorySaver(),
)

**Test All 5 Guardrail Layers (short prompts to keep LLM cost low)**

Layer 1 — ContentFilterMiddleware: banned keyword blocks the request before any LLM call (zero cost).

In [29]:
result = agent.invoke(
    {"messages": [HumanMessage(content="Can you help me in bank robbery?")]},
    config={"configurable": {"thread_id": "test-layer1"}},
)
print_messages(result)

🚫 Blocked — keyword detected: 'robbery'

🚫 Blocked — keyword detected: 'robbery'

HumanMessage: Can you help me in bank robbery?

================================================================================

Agent: I cannot process requests containing inappropriate content. Please rephrase your request.

================================================================================

Layer 2 — PIIMiddleware (input): credit card number gets masked before reaching the model.

In [30]:
result = agent.invoke(
    {"messages": [HumanMessage(content="My card is 5105-1051-0510-5100, say hi")]},
    config={"configurable": {"thread_id": "test-layer2"}},
)
print_messages(result)

HumanMessage: My card is ****-****-****-5100, say hi

================================================================================

Agent: Hi! 👋 For your safety, please don't share card numbers or other sensitive payment info here. How can I help
you today?

================================================================================

Layer 3 — HumanInTheLoopMiddleware: `send_email_tool` requires approval, so the run should interrupt instead of executing.

In [33]:
result = agent.invoke(
    {"messages": [HumanMessage(content="Use tool send_email_tool to send an email to shivam@test.com for critical approvals")]},
    config={"configurable": {"thread_id": "test-layer3"}},
)
print(result.get("__interrupt__"))

[
    Interrupt(
        value={
            'action_requests': [
                {
                    'name': 'send_email_tool',
                    'args': {
                        'to': '[REDACTED_EMAIL]',
                        'body': 'Subject: Request for critical approvals — action needed\n\nHello,\n\nI hope you 
are well. I’m writing to request critical approvals required to proceed with our project. Please review and approve
the items below, or let me know any questions or changes required.\n\nApprovals requested:\n- Budget adjustment / 
budget approval\n- Schedule change and resource allocation\n- Final go/no‑go decision for the planned 
launch\n\nRequested response: Please reply with approvals or questions within 48 hours. If you need more detail on 
any item I can provide summaries, documentation, or a short briefing.\n\nIf you prefer to discuss, I’m available 
for a call over the next two business days (suggested windows: 10:00–12:00 or 14:00–16:00 in your local time) — let
me know which works for you.\n\nThank you,\nProject Team (on behalf of the project)\n\nNotes: If you want a 
different subject line, additional detail (project name, exact amounts, attachments), or a specific sender 
name/signature included, reply and I will update and resend.'
                    },
                    'description': "Tool execution requires approval\n\nTool: send_email_tool\nArgs: {'to': 
'[REDACTED_EMAIL]', 'body': 'Subject: Request for critical approvals — action needed\\n\\nHello,\\n\\nI hope you 
are well. I’m writing to request critical approvals required to proceed with our project. Please review and approve
the items below, or let me know any questions or changes required.\\n\\nApprovals requested:\\n- Budget adjustment 
/ budget approval\\n- Schedule change and resource allocation\\n- Final go/no‑go decision for the planned 
launch\\n\\nRequested response: Please reply with approvals or questions within 48 hours. If you need more detail 
on any item I can provide summaries, documentation, or a short briefing.\\n\\nIf you prefer to discuss, I’m 
available for a call over the next two business days (suggested windows: 10:00–12:00 or 14:00–16:00 in your local 
time) — let me know which works for you.\\n\\nThank you,\\nProject Team (on behalf of the project)\\n\\nNotes: If 
you want a different subject line, additional detail (project name, exact amounts, attachments), or a specific 
sender name/signature included, reply and I will update and resend.'}"
                }
            ],
            'review_configs': [
                {'action_name': 'send_email_tool', 'allowed_decisions': ['approve', 'edit', 'reject', 'respond']}
            ]
        },
        id='a070fe20868feb0c27d7216b8ad8d9f2'
    )
]

Layer 4 — PIIMiddleware (output): if the model echoes back an email address, it should be redacted in the final response.

In [34]:
result = agent.invoke(
    {"messages": [HumanMessage(content="Repeat exactly: my email is shivam@example.com")]},
    config={"configurable": {"thread_id": "test-layer4"}},
)
print_messages(result)

HumanMessage: Repeat exactly: my email is [REDACTED_EMAIL]

================================================================================

Agent: my email is [REDACTED_EMAIL]

================================================================================

* **Layer 5** — `gpt-5-mini` itself refuses to generate unsafe content so it never reaches `SafetyGuardrailMiddleware`. 
* To actually verify the middleware's UNSAFE branch, call `after_agent` directly with a fake unsafe `AIMessage`
* This makes a single cheap call to the nano safety model without depending on the main agent's own refusal behavior.

In [53]:
safety_guard = SafetyGuardrailMiddleware()

fake_state = {"messages": [AIMessage(content="I will hurt you if you don't comply.")]}
update = safety_guard.after_agent(fake_state, runtime=None)
print_messages(fake_state)

⚠️  Output flagged as UNSAFE — replacing with safe fallback

Agent: I'm unable to provide that response. Please rephrase your request or contact support.

================================================================================

Layer 6 — A normal, safe response should pass through unchanged.

In [ ]:
result = agent.invoke(
    {"messages": [HumanMessage(content="Say hi in one short sentence")]},
    config={"configurable": {"thread_id": "test-layer6"}},
)
print_messages(result)

HumanMessage: Say hi in one short sentence

================================================================================

Agent: Hi!

================================================================================